In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import datetime as dt
from functools import partial
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from scipy.optimize import curve_fit

from piepy.psychophysics.wheel.detection.wheelDetectionExperimentHub import WheelDetectionExperimentHub
from piepy.core.data_functions import make_subsets

from piepy.psychophysics.wheel.detection.wheelDetectionGroupedAggregator import WheelDetectionGroupedAggregator
from piepy.plotters.plotting_utils import set_style

from piepy.psychophysics.fit_funcs import mle_fit, neg_likelihood, weibull, erf_psycho

In [ ]:
DATA_PATH = Path.cwd().parents[0] / "260410_Ncomm_inhibition_data.parquet"

CONTRAST_IDX = {0.0:0,
                0.125:1,
                0.5:2}

MANIP_LABEL = {0:"Control",
               1:"CNO",
               2:"Laser ON",
               3:"Laser ON + CNO"}

MANIP_MARKER = {0:"o",
                1:"s",
                2:"x",
                3:"X"}

STYPE_COLOR = {"0.04cpd_8.0Hz":"#FF7F0F",
               "0.16cpd_0.5Hz":"#0099C2"}

In [ ]:
def nan_generator(shape:tuple):
    """_summary_

    Args:
        shape (tuple): _description_

    Returns:
        _type_: _description_
    """
    x = np.zeros(shape)
    x[:] = np.nan
    return x

def sigmoid(x_arr, x0, L, k, p_false=0):
    """
    x0 = x value of the sigmoid midpoint
    L = max value of sigmoid
    k = logistic growth rate aka steepness
    b = y-offset(if needed)
    """
    return L / (1.0 + np.exp(-k * (x_arr - x0))) + p_false

def naka_rushton(c_arr,c50,n,p_max,p_false=0):
    """
    c_arr : contrast array
    n : slope/shape parameter
    c50 : contrast giving halfway point between p_false and p_max
    p_false : probability of saying “yes” when no stimulus is present
    p_max : hit rate at high contrast
    """
    return p_false + (p_max-p_false)/(1+(c50/c_arr)**n)

def fit_naka_rushton(lin_ax, responses, p_false, bounds=None):
    
    fit_func = naka_rushton
    if bounds is None:
        bounds = ([0.5, 1, 0], [1.5, 5, 1]) # [c50,n,p_max]
    
    fit_func_partial = partial(fit_func,p_false=p_false)
    popt,pcov = curve_fit(fit_func_partial, 
                            xdata=lin_ax, 
                            ydata=responses,
                            p0=[1,3,0], 
                            bounds=bounds, 
                            maxfev=100)
    interp_c50 = np.interp(popt[0],lin_ax,[0,12.5,50])
    fitted_p_max = popt[2]
    return popt, interp_c50, fitted_p_max
    
def bootstrap_fit(lin_ax,responses,n_boot=1000):
    
    

    params = np.zeros((n_boot,3))
    params[:] = np.nan
    interp_c50_array = []
    for i in range(n_boot):
        resampled = np.apply_along_axis(np.random.choice,0,responses,size=responses.shape[0],replace=True)
        means = np.nanmean(resampled,axis=0)
        popt,interp_c50,fitted_p_max = fit_naka_rushton(lin_ax,means,p_false=means[0])
        params[i,:] = popt
        interp_c50_array.append(interp_c50)
    return params, interp_c50_array

def bootstrap_means(lin_ax,responses,n_boot=1000):
    
    means = np.zeros((n_boot,3))
    means[:] = np.nan
    interp_c50_array = []
    for i in range(n_boot):
        resampled = np.apply_along_axis(np.random.choice,0,responses,size=responses.shape[0],replace=True)
        means[i,:]=np.nanmean(resampled,axis=0)
        
    return means

In [ ]:
hub = WheelDetectionExperimentHub()
# load from session list
# hub.set_data(all_sessions,
#              load_sessions=True,
#              make_summary=True)
# hub.data.write_parquet("250206_experiment_data.parquet")

# load from saved data
all_data = pl.read_parquet(DATA_PATH)
hub.set_data(all_data,
             load_sessions=True,
             make_summary=True)

In [ ]:
areas = ["V1", "HVA", "dorsal", "ventralPM"]
stim_combination = "0.04cpd_8.0Hz+0.16cpd_0.5Hz"
all_data = {}
for a in areas:
    df_cno = hub.filter_by_areas(areas=[a],
                                 stim_combination=stim_combination,
                                 isCNO=True,
                                 strict_performance=False)

    dff_cno = hub.filter_by_animals(['KC147','KC148','KC149','KC151','KC152'],
                                    data=df_cno,
                                    stim_combination=stim_combination,
                                    isCNO=True)

    df_sal = hub.filter_by_areas(areas=[a],
                                 stim_combination=stim_combination,
                                 isCNO=False,
                                 strict_performance=False)

    dff_sal = hub.filter_by_animals(['KC147','KC148','KC149','KC151','KC152'],
                                    data=df_sal,
                                    stim_combination=stim_combination,
                                    isCNO=False)


    df = pl.concat([dff_cno,dff_sal])
    all_data[a] = df
all_data

In [ ]:
aggregator = WheelDetectionGroupedAggregator()

area_matrices = {}
for k,v in all_data.items():
    aggregator.set_data(data=v)
    aggregator.group_data(
        group_by=[
            "stim_side",
            "animalid",
            "contrast",
            "isCNO",
            "opto_pattern",
            "stim_type", 
        ]
    )
    aggregator.calculate_hit_rates()
    aggregator.calculate_opto_pvalues()
    
    qq = (aggregator.grouped_data
         .filter((pl.col("contrast") == 0) & (pl.col("opto_pattern") == -1))
         .group_by(["animalid", "isCNO"])
         .agg([
             pl.col("hit_count").sum(),
             pl.col("count").sum()
         ])
    ).sort("animalid","isCNO")
    catch_trials = qq.with_columns(
        (pl.col("hit_count") / pl.col("count")).alias("baseline_hr")
    )
    
    _data = aggregator.grouped_data.join(
        catch_trials.select(["animalid", "isCNO","baseline_hr"]),
        how="inner",
        left_on= ["animalid", "isCNO"],
        right_on=["animalid", "isCNO"],
    )
    
    plot_data = _data.drop_nulls(["stim_side"]).filter(pl.col("stim_side") != "ipsi")

    
    q = (plot_data.group_by(["isCNO","stim_type","contrast","opto_pattern"])
            .agg(
                [pl.col("animalid"),
                 pl.col("hit_rate"),
                 pl.col("count"),
                 pl.col("baseline_hr"),
                 pl.col("median_hit_reaction_times")
                ]
            )).sort("isCNO","stim_type","contrast","opto_pattern")
    
    
    data_mat = nan_generator((
        plot_data.n_unique("animalid"),     # 
        plot_data.n_unique("contrast"),     # contrasts
        4,                                  # control, CNO, opto, CNO+opto
        plot_data.n_unique("stim_type"),    # stim types
        2                                   # [hit_rate, count]
    ))
    
    for stim_tup in make_subsets(q,"stim_type",start_enumerate=0):
        stim_idx = stim_tup[0]
        stim_df = stim_tup[-1]
        for manip_tup in make_subsets(stim_df,["isCNO","opto_pattern"]):
            # CNO
            if manip_tup[0]:
                _is_cno = "1"
            else:
                _is_cno = "0"

            # opto
            if manip_tup[1]==0:
                _is_opto = "1"
            else:
                _is_opto = "0"
                
            manip_idx = int(_is_opto+_is_cno,2)
            manip_df = manip_tup[-1]
            for contrast_tup in make_subsets(manip_df,"contrast",start_enumerate=0):
                contrast = contrast_tup[1]
                contrast_idx = CONTRAST_IDX[contrast]
                contrast_df = contrast_tup[-1]
                
                hr = contrast_df["hit_rate"].explode().to_numpy()
                count = contrast_df["count"].explode().to_numpy()
                
                # b_hr = contrast_df["baseline_hr"].explode().to_numpy()
                
                # if contrast != 0:
                #     #percent_correct, floored at 0
                #     _pc = (hr - b_hr)/np.max(hr-b_hr)
                #     prcnt_corr = np.array([max(i,0) for i in _pc])
                # else:
                #     prcnt_corr = b_hr
                temp = np.hstack((hr.reshape(-1,1),
                                  count.reshape(-1,1)))
                data_mat[:,contrast_idx,manip_idx,stim_idx,:] = temp
                                                                
    area_matrices[k] = data_mat
                

## Fill non-existent opto 0% contrast hit rates with copies of correpsonding non-opto values

In [ ]:
filled_area_matrices = area_matrices.copy()
for k,v in filled_area_matrices.items():

# non-cno non-opto to non-cno opto
    v[:,0,2,:,:] = v[:,0,0,:,:]

    # cno non-opto to cno opto
    v[:,0,3,:,:] = v[:,0,1,:,:]
    
    filled_area_matrices[k] = v

## Plotting

In [ ]:
set_style("print") 
do_fit="naka-rushton"
n_boot = 1000
do_sem = True

cm = 1/2.54

f,axs = plt.subplots(2,
                     len(filled_area_matrices),
                     figsize=(50*cm,30*cm))

stypes = ["0.04cpd_8.0Hz","0.16cpd_0.5Hz"]
lin_ax = [0,1,2]
log_ax = [0.1,12.5,50]
do_log=True

xax = log_ax if do_log else lin_ax

custom_handles = []
for st_idx,stype in enumerate(stypes):
    for area,ax in zip(filled_area_matrices.keys(),axs[st_idx,:]):
        stype_slice = filled_area_matrices[area][:,:,:,st_idx,:]
        
        
        for m_idx in range(stype_slice.shape[2]): 
            manip_slice = stype_slice[:,:,m_idx,0] # use only hr and not count
            
            avg_hr = np.nanmean(manip_slice,axis=0)
            sem_hr = stats.sem(manip_slice,axis=0,nan_policy="omit")
            
            # params, interp_c50_array = bootstrap_fit(lin_ax,manip_slice,n_boot=n_boot)
            # means = bootstrap_means(lin_ax,manip_slice,n_boot=n_boot)
            boot_res = stats.bootstrap((manip_slice,),np.nanmean,vectorized=True,n_resamples=1000,paired=True,confidence_level=0.95)
            
            # _fit_data = np.vstack((np.array(lin_ax).reshape(1,-1), 
            #                        counts.reshape(1,-1), 
            #                        hr.reshape(1,-1)))
            # confs = np.percentile(means,[2.5,97.5],axis=0)
            

            popt = [None, None, None]
            interp_c50 = None
            is_fitted = True
            try:                
                if do_fit == "sigmoid":
                    fit_func = sigmoid
                    bounds = ([0.5, 0, 1], [1.5, 1, 5]) # [x0,L,k]
                    fit_func_partial = partial(fit_func,p_false=set_p_false)
                    popt,pcov = curve_fit(fit_func_partial, 
                                          xax, 
                                          hr, 
                                          p0=[1,0.5,3], 
                                          bounds=bounds, 
                                          maxfev=5000)
                    
                    interp_c50 = np.interp(popt[0],lin_ax,[0,0.125,0.5])
                    fitted_p_max = popt[1]
 
                elif do_fit == "naka-rushton":
                    fit_func = naka_rushton
                    bounds = ([1,1,0],[30,3,1]) # [c50,n,p_max]([0.5, 1, 0], [1.5, 5, 1])
                    popt_avg, interp_c50_avg,fitted_p_max_avg = fit_naka_rushton(xax,avg_hr,p_false=avg_hr[0],bounds=bounds)

                    
            except RuntimeError as e:
                is_fitted = False
            
            # jitter = np.random.random(len(lin_ax))*0.01
            # sc = ax.errorbar(lin_ax+jitter,
            #                  hr,
            #                  yerr=(hr+semm,hr-semm),
            #                  color="k",
            #                  linewidth=0,
            #                  elinewidth=1,
            #                  marker=MANIP_MARKER[col_idx],
            #                  label=f"{MANIP_LABEL[col_idx]} C50={round(interp_c50,2)}, pmax={round(fitted_p_max*100,2)}",
            #                  zorder=2,)
            
            # the actual avg dots
            sc = ax.scatter(xax,avg_hr,
                            color="k",
                            label=f"{MANIP_LABEL[m_idx]} C50={round(interp_c50_avg,2)}, pmax={round(fitted_p_max_avg*100,2)}",
                            marker=MANIP_MARKER[m_idx],
                            zorder=2)
            print("\n")
            print(area, stype,MANIP_LABEL[m_idx])
            print("AVG: ",avg_hr)
            print("boot+: ",boot_res.confidence_interval.low)
            print("boot-: ",boot_res.confidence_interval.high)
            print("sem:", sem_hr)
            
            if do_sem:
                _ymin = avg_hr - sem_hr
                _ymax = avg_hr + sem_hr
            else:
                _ymin = boot_res.confidence_interval.low
                _ymax = boot_res.confidence_interval.high
            
            ax.vlines(xax,ymin=_ymin,ymax=_ymax,
                      color="k")
            
            
            if area == "V1":
                ax.set_ylabel("Hit rate (%)")
            
            if st_idx == 0 and area=="V1":
                custom_handles.append(Line2D([], [], marker=MANIP_MARKER[m_idx], color='k', markerfacecolor='k', markersize=10, label=MANIP_LABEL[m_idx]))
            
            if is_fitted:
                x_fit1 = np.linspace(xax[0], xax[1], 50)
                x_fit2 = np.linspace(xax[1], xax[2], 50)
                x_fit = np.concatenate((x_fit1,x_fit2[1:]))
                # y_fit = weibull(params, x_fit)
                y_fit_avg = fit_func(x_fit, *popt_avg, p_false=avg_hr[0])
                ax.plot(x_fit,y_fit_avg,
                        linewidth=2,
                        color=STYPE_COLOR[stype],
                        zorder=0
                        )
                
                
                # y_fit_upper = fit_func(x_fit,*confs[1,:],p_false=0)
                # y_fit_lower = fit_func(x_fit,*confs[0,:],p_false=0)
                
                # ax.fill_between(x_fit,
                #                 y_fit_avg-y_fit_lower,
                #                 y_fit_avg+y_fit_upper,
                #                 alpha=0.2)
                
                ax.legend(loc="upper left",frameon=False,fontsize=10)
        
        if do_log:
            ax.set_xscale("log")
        ax.set_title(f"{stype}_{area}")
        ax.set_xlabel("Contrast (%)")
        # ax.set_xticks(lin_ax)
        # ax.set_xticklabels([0,12.5,50])
        ax.set_ylim([0,1])
        ax.set_xlim([0.5,100])
        ax.set_yticks([0,0.25,0.5,0.75,1.0])
        ax.set_yticklabels([0,25,50,75,100])
print(custom_handles)
f.suptitle(f"{do_fit}")
f.legend(handles=custom_handles, ncol=len(MANIP_MARKER),loc='lower center')